### 1. Checking out the data
First, I'm just listing the columns from both the US Hospital data and the Bangladesh (DiaBD) set. I need to see which columns they have in common so I can figure out how to merge them later.

In [ ]:
print("=== Diabetes 130-US Hospitals columns ===")
print(df_130.columns.tolist())
print("\n=== DiaBD Bangladesh columns ===")
print(df_diabd.columns.tolist())

=== Diabetes 130-US Hospitals columns ===
['race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

=== DiaBD Bangladesh columns ===
['age', 'gender', 'pulse_rate', 'systolic_bp', 'diastolic_bp', 'glucose', 'height', 'weight', 'bmi', 'family_diabetes', 'hypertensive', '

### 2. Squashing the datasets together
Since the two datasets have different column names, I'm renaming them to match (like turning 'Outcome' into 'diabetic'). I'm also adding a 'source' tag so I don't forget where each row came from once they're all in one big table.

In [ ]:
# Standardise PIMA
pima_clean = df_pima[['Age', 'Glucose', 'BMI', 'BloodPressure', 'Outcome']].copy()
pima_clean.columns = ['age', 'glucose', 'bmi', 'diastolic_bp', 'diabetic']
pima_clean['diabetic'] = pima_clean['diabetic'].map({1: 'Yes', 0: 'No'})
pima_clean['gender'] = 'Female'  # PIMA is female-only
pima_clean['source'] = 'PIMA'

# Standardise DiaBD
diabd_clean = df_diabd[['age', 'glucose', 'bmi', 'diastolic_bp', 'diabetic', 'gender']].copy()
diabd_clean['source'] = 'DiaBD'

# Combine
df_clinical = pd.concat([pima_clean, diabd_clean], ignore_index=True)

print("Combined clinical dataset:", df_clinical.shape)
print("\nSource breakdown:")
print(df_clinical['source'].value_counts())
print("\nTarget breakdown:")
print(df_clinical['diabetic'].value_counts())
df_clinical.head()

Combined clinical dataset: (6056, 7)

Source breakdown:
source
DiaBD    5288
PIMA      768
Name: count, dtype: int64

Target breakdown:
diabetic
No     5446
Yes     610
Name: count, dtype: int64


,age,glucose,bmi,diastolic_bp,diabetic,gender,source
0,50,148.0,33.6,72,Yes,Female,PIMA
1,31,85.0,26.6,66,No,Female,PIMA
2,32,183.0,23.3,64,Yes,Female,PIMA
3,21,89.0,28.1,66,No,Female,PIMA
4,33,137.0,43.1,40,Yes,Female,PIMA


### 3. Verifying Data Integrity
This cell checks for missing values and verifies data types to ensure the combined dataset is ready for preprocessing.

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df_clinical.isnull().sum())
print("\nData types:")
print(df_clinical.dtypes)

Missing values per column:
age             0
glucose         0
bmi             0
diastolic_bp    0
diabetic        0
gender          0
source          0
dtype: int64

Data types:
age               int64
glucose         float64
bmi             float64
diastolic_bp      int64
diabetic         object
gender           object
source           object
dtype: object


### 4. Data Cleaning and Train-Test Split
We handle 'hidden' missing values (zeros in PIMA) by imputing with medians, encode categorical variables, and perform a stratified split to maintain the class balance.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Fix PIMA's known issue: 0 values in glucose/bmi are actually missing, not real zeros
df_clinical.loc[df_clinical['glucose'] == 0, 'glucose'] = np.nan
df_clinical.loc[df_clinical['bmi'] == 0, 'bmi'] = np.nan
df_clinical.loc[df_clinical['diastolic_bp'] == 0, 'diastolic_bp'] = np.nan

# Fill any of those newly-created missing values with the median
df_clinical['glucose'] = df_clinical['glucose'].fillna(df_clinical['glucose'].median())
df_clinical['bmi'] = df_clinical['bmi'].fillna(df_clinical['bmi'].median())
df_clinical['diastolic_bp'] = df_clinical['diastolic_bp'].fillna(df_clinical['diastolic_bp'].median())

# Encode gender and target
df_clinical['gender_encoded'] = LabelEncoder().fit_transform(df_clinical['gender'])
df_clinical['target'] = df_clinical['diabetic'].map({'Yes': 1, 'No': 0})

# Features and target
feature_cols = ['age', 'glucose', 'bmi', 'diastolic_bp', 'gender_encoded']
X = df_clinical[feature_cols]
y = df_clinical['target']

# Stratified split — keeps the ~90/10 imbalance ratio consistent in both train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\nTraining target balance:")
print(y_train.value_counts())
print("\nTest target balance:")
print(y_test.value_counts())

Training set: (4844, 5)
Test set: (1212, 5)

Training target balance:
target
0    4356
1     488
Name: count, dtype: int64

Test target balance:
target
0    1090
1     122
Name: count, dtype: int64


### 5. Training Baseline Models (XGBoost & Random Forest)
We train two models using methods to handle the 9:1 class imbalance: `scale_pos_weight` for XGBoost and `class_weight='balanced'` for Random Forest.

In [ ]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# Calculate class weight ratio for imbalance handling
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
print(f"Class imbalance ratio (scale_pos_weight): {scale_pos_weight:.2f}")

# ---- XGBoost ----
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,  # handles the 9:1 imbalance
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

print("\n=== XGBoost Results ===")
print("Accuracy:", accuracy_score(y_test, xgb_pred))
print("AUC-ROC:", roc_auc_score(y_test, xgb_proba))
print("\n", classification_report(y_test, xgb_pred))

# ---- Random Forest ----
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',  # handles imbalance differently — built-in RF method
    random_state=42
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

print("\n=== Random Forest Results ===")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("AUC-ROC:", roc_auc_score(y_test, rf_proba))
print("\n", classification_report(y_test, rf_pred))

Class imbalance ratio (scale_pos_weight): 8.93

=== XGBoost Results ===
Accuracy: 0.8646864686468647
AUC-ROC: 0.8558429839073545

               precision    recall  f1-score   support

           0       0.96      0.89      0.92      1090
           1       0.40      0.67      0.50       122

    accuracy                           0.86      1212
   macro avg       0.68      0.78      0.71      1212
weighted avg       0.90      0.86      0.88      1212


=== Random Forest Results ===
Accuracy: 0.8943894389438944
AUC-ROC: 0.8625357196570912

               precision    recall  f1-score   support

           0       0.95      0.93      0.94      1090
           1       0.48      0.54      0.51       122

    accuracy                           0.89      1212
   macro avg       0.71      0.74      0.72      1212
weighted avg       0.90      0.89      0.90      1212



### 6. Ensemble Model
By averaging the predicted probabilities of XGBoost and Random Forest, we create an ensemble model to potentially improve the robustness of our predictions.

In [ ]:
# Simple averaging ensemble of XGBoost + Random Forest probabilities
ensemble_proba = (xgb_proba + rf_proba) / 2
ensemble_pred = (ensemble_proba >= 0.5).astype(int)

print("=== Ensemble (XGBoost + Random Forest) Results ===")
print("Accuracy:", accuracy_score(y_test, ensemble_pred))
print("AUC-ROC:", roc_auc_score(y_test, ensemble_proba))
print("\n", classification_report(y_test, ensemble_pred))

=== Ensemble (XGBoost + Random Forest) Results ===
Accuracy: 0.8861386138613861
AUC-ROC: 0.8642502631974733

               precision    recall  f1-score   support

           0       0.96      0.91      0.94      1090
           1       0.45      0.63      0.53       122

    accuracy                           0.89      1212
   macro avg       0.70      0.77      0.73      1212
weighted avg       0.91      0.89      0.89      1212



### 7. Feature Expansion (v2)
We create a more complex dataset by including features that are present in only one of the sources (e.g., pregnancies, pulse rate) to see if they improve predictive power.

In [ ]:
# Standardise PIMA with more features
pima_full = df_pima[['Age', 'Glucose', 'BMI', 'BloodPressure', 'Pregnancies',
                       'SkinThickness', 'Insulin', 'DiabetesPedigreeFunction', 'Outcome']].copy()
pima_full.columns = ['age', 'glucose', 'bmi', 'diastolic_bp', 'pregnancies',
                       'skin_thickness', 'insulin', 'pedigree_function', 'diabetic']
pima_full['diabetic'] = pima_full['diabetic'].map({1: 'Yes', 0: 'No'})
pima_full['gender'] = 'Female'
pima_full['systolic_bp'] = np.nan       # PIMA doesn't have this — will impute
pima_full['pulse_rate'] = np.nan
pima_full['family_diabetes'] = np.nan
pima_full['hypertensive'] = np.nan
pima_full['cardiovascular_disease'] = np.nan
pima_full['source'] = 'PIMA'

# Standardise DiaBD with more features
diabd_full = df_diabd[['age', 'glucose', 'bmi', 'diastolic_bp', 'systolic_bp',
                         'pulse_rate', 'gender', 'family_diabetes', 'hypertensive',
                         'cardiovascular_disease', 'diabetic']].copy()
diabd_full['pregnancies'] = np.nan       # DiaBD doesn't have this
diabd_full['skin_thickness'] = np.nan
diabd_full['insulin'] = np.nan
diabd_full['pedigree_function'] = np.nan
diabd_full['source'] = 'DiaBD'

# Combine
df_clinical_v2 = pd.concat([pima_full, diabd_full], ignore_index=True)

print("Expanded combined dataset:", df_clinical_v2.shape)
print("\nMissing values per column:")
print(df_clinical_v2.isnull().sum())

Expanded combined dataset: (6056, 16)

Missing values per column:
age                          0
glucose                      0
bmi                          0
diastolic_bp                 0
pregnancies               5288
skin_thickness            5288
insulin                   5288
pedigree_function         5288
diabetic                     0
gender                       0
systolic_bp                768
pulse_rate                 768
family_diabetes            768
hypertensive               768
cardiovascular_disease     768
source                       0
dtype: int64


### 8. Advanced Preprocessing with Missingness Indicators
Before imputing medians for the expanded feature set, we create 'missing flags' to preserve the information that certain data was originally absent for specific records.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create "was_missing" flags before imputing (these become useful features too)
features_with_gaps = ['pregnancies', 'skin_thickness', 'insulin', 'pedigree_function',
                        'systolic_bp', 'pulse_rate', 'family_diabetes', 'hypertensive',
                        'cardiovascular_disease']

for col in features_with_gaps:
    df_clinical_v2[f'{col}_missing'] = df_clinical_v2[col].isnull().astype(int)

# Impute missing values with median
for col in features_with_gaps:
    df_clinical_v2[col] = df_clinical_v2[col].fillna(df_clinical_v2[col].median())

# Fix PIMA's hidden zero-as-missing issue again for the expanded set
for col in ['glucose', 'bmi', 'diastolic_bp', 'skin_thickness', 'insulin']:
    mask = (df_clinical_v2[col] == 0) & (df_clinical_v2['source'] == 'PIMA')
    df_clinical_v2.loc[mask, col] = df_clinical_v2[df_clinical_v2['source']=='PIMA'][col].median()

# Encode categorical columns
df_clinical_v2['gender_encoded'] = LabelEncoder().fit_transform(df_clinical_v2['gender'])
df_clinical_v2['target'] = df_clinical_v2['diabetic'].map({'Yes': 1, 'No': 0})

# Final feature set
feature_cols_v2 = ['age', 'glucose', 'bmi', 'diastolic_bp', 'gender_encoded',
                     'pregnancies', 'skin_thickness', 'insulin', 'pedigree_function',
                     'systolic_bp', 'pulse_rate', 'family_diabetes', 'hypertensive',
                     'cardiovascular_disease'] + [f'{c}_missing' for c in features_with_gaps]

X2 = df_clinical_v2[feature_cols_v2]
y2 = df_clinical_v2['target']

print("Final feature count:", X2.shape[1])
print("Features used:", feature_cols_v2)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)
print("\nTraining set:", X2_train.shape)
print("Test set:", X2_test.shape)

Final feature count: 23
Features used: ['age', 'glucose', 'bmi', 'diastolic_bp', 'gender_encoded', 'pregnancies', 'skin_thickness', 'insulin', 'pedigree_function', 'systolic_bp', 'pulse_rate', 'family_diabetes', 'hypertensive', 'cardiovascular_disease', 'pregnancies_missing', 'skin_thickness_missing', 'insulin_missing', 'pedigree_function_missing', 'systolic_bp_missing', 'pulse_rate_missing', 'family_diabetes_missing', 'hypertensive_missing', 'cardiovascular_disease_missing']

Training set: (4844, 23)
Test set: (1212, 23)
